# Ask CIP v7: Ask's own reader, retrained on checked labels (no AI, no teacher here)

v6 taught a small model (Ask's reader, `models/ask-reader`) Qwen3-32B's picks for 12,281 practice questions. v7 keeps those
picks but replaces the 1,000 likeliest wrong ones with labels checked by Claude in the build session (no API key), and adds
238 questions no survey answers, labelled "none of these". No large model runs here, so a small GPU will do.

1. **The labels** come in `v7-train.json` (v6's practice questions with each one's label, made on the Mac).
2. **The students**: MiniLM L6 (today's size) and L4 (smaller, about twice as fast) learn them; "none of these" is a choice.
3. **Shrunk to 8-bit**, checked with the server's own code, timed on one CPU thread; `ask-v7.zip` downloads: send it back
   (scored on the Mac inside Ask with no AI).

**Run:** Runtime → Change runtime type → **L4** (T4 works, slower) → Runtime → Run all → pick `v7-train.json` and
`v6-tests.json` from `Downloads/cip-colab`. About 1-1½ hours on an L4; keep the laptop open. No API key is used anywhere.


In [ ]:
# 1. Libraries (PyTorch and transformers come with Colab), the GPU, the files and their labels
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)
import json, os, random, shutil, time
import numpy as np, torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → L4, then Run all again"
print(torch.cuda.get_device_name(0))
if not (os.path.exists("v7-train.json") and os.path.exists("v6-tests.json")):
    from google.colab import files
    print("Pick v7-train.json and v6-tests.json from Downloads/cip-colab (hold ⌘)")
    files.upload()
data, tests = json.load(open("v7-train.json")), json.load(open("v6-tests.json"))
labels = [i["label"] for i in data["items"]]   # a candidate, None for "none of these", -1 for left out
print(f"{len(labels)} practice questions · {sum(p is None for p in labels)} 'none of these' · {sum(p == -1 for p in labels)} left out · "
      f"tests: {len(tests['golden'])} golden, {len(tests['bank'])} big exam")


In [ ]:
# 3. The student: for one question, a score per candidate line (the line the AI reads, without its number); "none of
# these" is a fixed score of 0, so a candidate is picked only when it scores above 0. Trained on the labels
from transformers import AutoModelForSequenceClassification, AutoTokenizer
DEVICE = "cuda"
def line_text(line):
    return line.split(". ", 1)[1] if ". " in line[:5] else line
examples = [(i["q"], [line_text(l) for l in i["lines"]], p) for i, p in zip(data["items"], labels) if p != -1]
random.Random(5).shuffle(examples)
held, train_ex = examples[:900], examples[900:]

def scores(model, tok, question, lines):
    enc = tok([question] * len(lines), lines, truncation=True, max_length=192, padding=True, return_tensors="pt").to(DEVICE)
    return model(**enc).logits.view(-1)

def choose(s):
    best = int(torch.argmax(s))
    return best if float(s[best]) > 0 else None

def agreement(model, tok, rows):
    model.eval()
    with torch.no_grad():
        return float(np.mean([choose(scores(model, tok, q, l)) == p for q, l, p in rows]))

def train(name, epochs=3, per_step=16, lr=3e-5):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=1).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * (len(train_ex) // per_step)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / (steps // 10 + 1)) * max(0.0, 1 - s / steps))
    best, best_state = -1.0, None
    for epoch in range(epochs):
        model.train(); random.Random(epoch).shuffle(train_ex); started = time.time()
        for s in range(0, len(train_ex) - per_step + 1, per_step):
            loss = 0
            for q, lines, p in train_ex[s:s + per_step]:   # softmax over "none" (0) and every candidate
                logits = torch.cat([torch.zeros(1, device=DEVICE), scores(model, tok, q, lines)])
                loss = loss + torch.nn.functional.cross_entropy(logits.unsqueeze(0), torch.tensor([0 if p is None else p + 1], device=DEVICE))
            (loss / per_step).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad()
        now = agreement(model, tok, held)
        print(f"  {name} epoch {epoch + 1}: agrees with the labels on {now:.1%} of held-out questions · {time.time() - started:.0f}s")
        if now > best:
            best, best_state = now, {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model.eval(), tok, best

students = {name: train(name) for name in ("cross-encoder/ms-marco-MiniLM-L6-v2", "cross-encoder/ms-marco-MiniLM-L4-v2")}

In [ ]:
# 4. Each student shrunk to the server's format (8-bit ONNX), read with the server's own code (api/ask_model.py read_pick),
# timed on one CPU thread (Colab's CPU is slower than the server's: both students go back, and the Mac times and scores
# them inside Ask with no AI: scripts/ask_v6.py score)
import onnxruntime, tokenizers
from onnxruntime.quantization import quantize_dynamic, QuantType
NAMES = ["input_ids", "attention_mask", "token_type_ids"]
options = onnxruntime.SessionOptions(); options.intra_op_num_threads = options.inter_op_num_threads = 1

def shrink(model, tok, folder):
    os.makedirs(folder, exist_ok=True)
    dummy = tok(["a question"], ["a survey question"], return_tensors="pt")
    torch.onnx.export(model.cpu().eval(), tuple(dummy[k] for k in NAMES), "full.onnx", input_names=NAMES, output_names=["logits"],
                      dynamic_axes={k: {0: "batch", 1: "tokens"} for k in NAMES} | {"logits": {0: "batch"}}, opset_version=17, dynamo=False)
    quantize_dynamic("full.onnx", f"{folder}/model_optimized.onnx", weight_type=QuantType.QInt8)
    tok.save_pretrained("tok_tmp"); shutil.copy("tok_tmp/tokenizer.json", f"{folder}/tokenizer.json")
    model.to(DEVICE)
    words = tokenizers.Tokenizer.from_file(f"{folder}/tokenizer.json"); words.enable_truncation(192); words.enable_padding(pad_id=0, pad_token="[PAD]")
    session = onnxruntime.InferenceSession(f"{folder}/model_optimized.onnx", options, providers=["CPUExecutionProvider"])
    def server_pick(question, lines):   # api/ask_model.py read_pick, line for line, then above 0.5 or "none"
        coded = words.encode_batch([(question, t) for t in lines]); ids = np.array([c.ids for c in coded], np.int64)
        feed = {"input_ids": ids, "attention_mask": np.array([c.attention_mask for c in coded], np.int64), "token_type_ids": np.array([c.type_ids for c in coded], np.int64)}
        p = 1 / (1 + np.exp(-session.run(None, feed)[0].reshape(-1)))
        return int(np.argmax(p)) if p.max() > 0.5 else None
    return server_pick

def proxy(test, picks):   # as v6's teacher.py scored it: the golden phrase or the big exam's key
    ok = 0
    for i, p in zip(tests[test], picks):
        if test == "golden":
            ok += (p is None) if i["want"] == "none" else (p is not None and p >= 0 and any(w in i["texts"][p].lower() for w in i["want"]))
        else:
            ok += p is not None and p >= 0 and i["keys"][p] == i["answer_key"]
    return ok

results, picks_by = {}, {}
for name, (model, tok, agree) in students.items():
    short = name.split("-")[-2]
    server_pick = shrink(model, tok, f"ask-v7/ask-reader-{short}")
    started = time.time()
    picks = {t: [server_pick(i["q"], [line_text(l) for l in i["lines"]]) for i in tests[t]] for t in ("golden", "bank")}
    ms = (time.time() - started) / sum(len(tests[t]) for t in ("golden", "bank")) * 1000
    same = float(np.mean([server_pick(q, l) == p for q, l, p in held[:300]]))
    results[short] = {"agrees with the labels": agree, "8-bit agrees": same, "ms a question, one CPU thread": round(ms),
                      **{t: proxy(t, picks[t]) for t in ("golden", "bank")}}
    print(f"{short}: agrees with the labels {agree:.1%} (8-bit {same:.1%}) · golden {results[short]['golden']} · big exam "
          f"{results[short]['bank']} · {ms:.0f} ms a question")
    picks_by[short] = picks

In [ ]:
# 5. The zip: both students (each a server models/ask-reader folder), their picks for the tests as Ask's AI
# replies (scored on the Mac with no AI) and the numbers
import zipfile
def as_reply(p):
    return json.dumps({"q": p, "r": None, "a": [], "k": "share", "g": "auto", "c": "none", "p": "all"})
for short in picks_by:
    json.dump({"sure": 2.0, "length": 192, "reads": "api/ask_model.py describe_candidate, the AI's line without its number"}, open(f"ask-v7/ask-reader-{short}/calibration.json", "w"))
for who, picks in [(f"v7-{short}", p) for short, p in picks_by.items()]:
    for t in ("golden", "bank"):
        json.dump({i["user"]: as_reply(p) for i, p in zip(tests[t], picks[t]) if p != -1}, open(f"ask-v7/replies-{who}-{t}.json", "w"))
json.dump(results, open("ask-v7/results.json", "w"), indent=1)
with zipfile.ZipFile("ask-v7.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, names in os.walk("ask-v7"):
        for n in names:
            z.write(os.path.join(root, n))
print(json.dumps(results, indent=1))
from google.colab import files
files.download("ask-v7.zip")
print("Done: send ask-v7.zip. Runtime → Disconnect and delete runtime stops the units.")